# datasetbaru

In [69]:
data = pd.read_csv(r'/kaggle/input/datasets/rizqiabroori/data-set-fertilizerbaru/fertilizer_recommendation_dataset.csv')
data

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,Loamy Soil,rice,Compost,Enhances organic matter and improves soil stru...
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,Peaty Soil,rice,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3,26.241975,0.755095,212.703513,6.883367,78.033687,151.012521,153.005712,1.517556,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
4,21.490157,0.730672,268.786767,7.578760,71.765123,66.257371,97.000886,1.782985,Peaty Soil,rice,Organic Fertilizer,"Enhances fertility naturally, ideal for peaty ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
3095,23.486430,0.531191,46.412228,6.733584,56.534283,146.111078,81.389366,1.602913,Neutral Soil,watermelon,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3096,24.289508,0.736699,63.068103,6.372709,56.358005,49.003277,46.695889,1.473656,Peaty Soil,watermelon,DAP,"Rich in phosphorus, essential for root develop..."
3097,23.945488,0.520513,41.344590,7.051515,55.738905,148.567285,90.057021,1.455045,Neutral Soil,watermelon,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3098,26.368604,0.547436,33.106012,6.615922,57.711705,96.662953,59.531473,0.614487,Acidic Soil,watermelon,Compost,Enhances organic matter and improves soil stru...


In [70]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

encoders = {}

categorical_cols = ['Soil', 'Crop']
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

target_col = 'Fertilizer'
le_target = LabelEncoder()
data[target_col] = le_target.fit_transform(data[target_col])
encoders['target'] = le_target

feature_cols = [
    'Temperature', 'Moisture', 'Rainfall', 'PH',
    'Soil', 'Crop',
    'Nitrogen', 'Potassium', 'Phosphorous', 'Carbon'
]

X = data[feature_cols]
y = data[target_col]

feature_order = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [71]:
X_train

,Temperature,Moisture,Rainfall,PH,Soil,Crop,Nitrogen,Potassium,Phosphorous,Carbon
2265,28.970318,0.389483,123.089973,5.766251,0,19,63.560720,56.566806,53.862910,1.735917
833,28.487843,0.789917,124.801917,6.995936,2,4,63.622055,129.744705,131.689625,1.355082
1895,32.413062,0.769942,92.961913,7.069078,2,1,58.015300,103.491629,134.173717,1.745716
980,21.678022,0.583813,79.681973,6.979950,3,5,59.284712,147.402742,142.866464,0.662365
2036,29.221729,0.470956,89.392447,4.698779,0,13,58.529698,45.309153,31.279845,1.410122
...,...,...,...,...,...,...,...,...,...,...
907,25.712983,0.647790,74.423888,6.487457,0,5,55.535678,55.986209,61.472849,1.110723
2055,35.970775,0.552259,196.221877,5.621210,0,13,60.723229,58.255611,32.792383,1.728056
823,22.413049,0.406487,175.203334,6.443273,0,4,65.099004,76.710525,66.663369,1.297313
2768,31.891353,0.445200,127.698563,8.828433,0,25,69.770829,63.695261,79.206208,2.369871


In [72]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': randint(150, 300),
    'max_depth': [15, 20, 25, None],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 4)
}

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

best_model = random_search.best_estimator_

print("Best Params:", random_search.best_params_)

Best Params: {'max_depth': 25, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 167}


In [73]:
from sklearn.metrics import f1_score, classification_report

preds = best_model.predict(X_test_scaled)

f1 = f1_score(y_test, preds, average='weighted')
print(f"F1 Score: {f1:.4f}")

target_names = encoders['target'].classes_.astype(str)
print(classification_report(y_test, preds, target_names=target_names))

F1 Score: 0.9934
                            precision    recall  f1-score   support

   Balanced NPK Fertilizer       0.97      1.00      0.98        31
                   Compost       1.00      1.00      1.00        75
                       DAP       1.00      1.00      1.00       211
General Purpose Fertilizer       1.00      1.00      1.00         6
                    Gypsum       1.00      0.82      0.90        11
                      Lime       1.00      1.00      1.00        36
         Muriate of Potash       0.98      1.00      0.99        65
        Organic Fertilizer       0.95      1.00      0.97        19
                      Urea       1.00      0.97      0.98        31
Water Retaining Fertilizer       0.99      0.99      0.99       135

                  accuracy                           0.99       620
                 macro avg       0.99      0.98      0.98       620
              weighted avg       0.99      0.99      0.99       620



In [74]:
import pandas as pd

importances = best_model.feature_importances_
feature_names = X_train.columns

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp)

       feature  importance
8  Phosphorous    0.317699
3           PH    0.137734
9       Carbon    0.137686
7    Potassium    0.135117
1     Moisture    0.096736
6     Nitrogen    0.092625
4         Soil    0.042197
2     Rainfall    0.014103
5         Crop    0.013129
0  Temperature    0.012974


In [75]:
print("Best Params:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

Best Params: {'max_depth': 25, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 167}
Best CV Score: 0.9916011615631645


In [76]:
print(f"F1 Score: {f1:.4f}")
print(classification_report(y_test, preds, target_names=target_names))

F1 Score: 0.9934
                            precision    recall  f1-score   support

   Balanced NPK Fertilizer       0.97      1.00      0.98        31
                   Compost       1.00      1.00      1.00        75
                       DAP       1.00      1.00      1.00       211
General Purpose Fertilizer       1.00      1.00      1.00         6
                    Gypsum       1.00      0.82      0.90        11
                      Lime       1.00      1.00      1.00        36
         Muriate of Potash       0.98      1.00      0.99        65
        Organic Fertilizer       0.95      1.00      0.97        19
                      Urea       1.00      0.97      0.98        31
Water Retaining Fertilizer       0.99      0.99      0.99       135

                  accuracy                           0.99       620
                 macro avg       0.99      0.98      0.98       620
              weighted avg       0.99      0.99      0.99       620



In [77]:
import joblib

joblib.dump(best_model, "/kaggle/working/random_forest_model.pkl")

['/kaggle/working/random_forest_model.pkl']

In [64]:
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoders, "encoders.pkl")
joblib.dump(feature_order, "feature_order.pkl")

['feature_order.pkl']